# 总体方差的估计完整教程：卡方分布与置信区间

## 📚 教学目标
1. 理解卡方分布的特征
2. 掌握方差的置信区间公式
3. 用 scipy.stats.chi2 计算临界值
4. 构建标准差的置信区间
5. 理解方差估计的特殊性

**参考书**：《基础统计学(第14版)》（Triola）第 7-3 节
**教学策略**：先用极小数据集理解卡方分布，再手算方差的置信区间，最后用 scipy 验证

---

## 1. 场景设定：为什么需要估计方差？

### 🎯 一个直觉问题

假设你是一家工厂的质量控制工程师：
- 你不仅关心产品的**平均尺寸**是否合格
- 你更关心产品的**尺寸一致性**（即方差或标准差）

方差太大会导致产品不合格，即使均值是正确的！

### 📖 书中的定义

> 样本方差 $s^2$ 是总体方差 $\sigma^2$ 的最佳点估计。
> 样本标准差 $s$ 是总体标准差 $\sigma$ 的最佳点估计。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print('✅ 库导入完成')

---

## 2. 卡方分布的认识

### 📐 卡方分布的定义

如果从一个正态分布总体（方差为 $\sigma^2$）中随机选取样本量为 $n$ 的样本，且每个样本有样本方差 $s^2$，那么统计量：

$$\chi^2 = \frac{(n-1)s^2}{\sigma^2}$$

服从自由度为 $df = n-1$ 的卡方分布。

### 📐 卡方分布的特征

| 特征 | 说明 |
|------|------|
| 形状 | 右偏态（不对称） |
| 取值范围 | 非负数（$\chi^2 \geq 0$） |
| 参数 | 自由度 $df$ |
| 均值 | $df$ |
| 随着 $df$ 增大 | 趋近于正态分布 |

### 📖 书中的要点

> 不同于正态分布和 t 分布，卡方分布呈右偏态。卡方值为非负数。卡方分布随着自由度的不同而不同。随着自由度的增大，卡方分布趋近于正态分布。

In [ ]:
# ========== 可视化：不同自由度的卡方分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：不同自由度的卡方分布
ax1 = axes[0]
dfs = [1, 2, 5, 10, 20]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

for df, color in zip(dfs, colors):
    x = np.linspace(0, max(40, df + 4*np.sqrt(2*df)), 1000)
    y = stats.chi2.pdf(x, df)
    ax1.plot(x, y, color=color, linewidth=2, label=f'df={df}')

ax1.set_xlabel('Chi-Square Value', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('Chi-Square Distribution with Different df', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 0.5)

# 右图：df=10 的卡方分布，标注临界值
ax2 = axes[1]
df_demo = 10
x_demo = np.linspace(0, 30, 1000)
y_demo = stats.chi2.pdf(x_demo, df_demo)
ax2.plot(x_demo, y_demo, 'b-', linewidth=2)
ax2.fill_between(x_demo, 0, y_demo, alpha=0.1, color='steelblue')

# 标注临界值
chi2_left = stats.chi2.ppf(0.025, df_demo)
chi2_right = stats.chi2.ppf(0.975, df_demo)
ax2.axvline(x=chi2_left, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.axvline(x=chi2_right, color='red', linestyle='--', linewidth=1.5, alpha=0.7)

# 标注面积
x_left = np.linspace(0, chi2_left, 100)
ax2.fill_between(x_left, 0, stats.chi2.pdf(x_left, df_demo), alpha=0.3, color='#e74c3c', label=f'Left 2.5%: χ²={chi2_left:.2f}')
x_right = np.linspace(chi2_right, 30, 100)
ax2.fill_between(x_right, 0, stats.chi2.pdf(x_right, df_demo), alpha=0.3, color='#e74c3c', label=f'Right 2.5%: χ²={chi2_right:.2f}')

ax2.set_xlabel('Chi-Square Value', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title(f'Chi-Square Distribution (df={df_demo}) with Critical Values', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  左图：卡方分布是右偏的，自由度越大越对称')
print(f'  右图：df=10 时，两侧各 2.5% 的临界值分别为 {chi2_left:.2f} 和 {chi2_right:.2f}')

---

## 3. 置信区间的核心公式

### 📐 总体方差的置信区间

$$\frac{(n-1)s^2}{\chi^2_R} < \sigma^2 < \frac{(n-1)s^2}{\chi^2_L}$$

### 📐 总体标准差的置信区间

$$\sqrt{\frac{(n-1)s^2}{\chi^2_R}} < \sigma < \sqrt{\frac{(n-1)s^2}{\chi^2_L}}$$

变量说明：
- $n$ = 样本量
- $s^2$ = 样本方差
- $\chi^2_R$ = 右侧临界值（分隔右侧面积 $\alpha/2$）
- $\chi^2_L$ = 左侧临界值（分隔左侧面积 $\alpha/2$）
- $df = n - 1$ = 自由度

### 📐 条件

1. 样本为简单随机样本
2. **总体必须服从正态分布**（即使是大样本！）

### ⚠️ 注意

对方差估计的正态性要求比均值估计**更严格**。与正态分布的较大偏差会导致较大的误差。

### 📖 书中的要点

> 构建方差或标准差的置信区间时，对正态分布的要求比前几节要严格得多。即使是大样本，总体也必须服从正态分布。

---

## 4. 微型数据手算：心率数据

### 📖 书中例题

成年女性心率（bpm，次/分钟）的简单随机样本：

76, 76, 86, 74, 66, 62, 78, 68, 62, 62, 74, 80, 54, 74, 74, 84, 60, 52, 84, 66, 56, 66

**问题**：构建总体标准差 $\sigma$ 的 95% 置信区间。

In [ ]:
# ========== 微型数据集 ==========
heart_rates = np.array([76, 76, 86, 74, 66, 62, 78, 68, 62, 62, 74,
                        80, 54, 74, 74, 84, 60, 52, 84, 66, 56, 66])
n_micro = len(heart_rates)
confidence_level = 0.95

print('📋 微型数据集：成年女性心率（bpm）')
print(f'  数据: {heart_rates}')
print(f'  样本量 n = {n_micro}')
print(f'  置信水平 = {confidence_level*100:.0f}%')

In [ ]:
# ========== 步骤 1: 计算样本统计量 ==========
s2_micro = np.var(heart_rates, ddof=1)  # 样本方差
s_micro = np.std(heart_rates, ddof=1)  # 样本标准差
df_micro = n_micro - 1

print(f'📊 步骤 1: 计算样本统计量')
print(f'  样本方差 s² = {s2_micro:.4f}')
print(f'  样本标准差 s = {s_micro:.4f}')
print(f'  自由度 df = n - 1 = {n_micro} - 1 = {df_micro}')

In [ ]:
# ========== 步骤 2: 确定卡方临界值 ==========
alpha_micro = 1 - confidence_level
chi2_L = stats.chi2.ppf(alpha_micro / 2, df_micro)    # 左侧临界值
chi2_R = stats.chi2.ppf(1 - alpha_micro / 2, df_micro) # 右侧临界值

print(f'📊 步骤 2: 确定卡方临界值')
print(f'  α = {alpha_micro}, α/2 = {alpha_micro/2}')
print(f'  左侧临界值 χ²_L = χ²_{{{1-alpha_micro/2:.3f}, {df_micro}}} = {chi2_L:.4f}')
print(f'    （分隔左侧面积 {alpha_micro/2} 的值）')
print(f'  右侧临界值 χ²_R = χ²_{{{alpha_micro/2:.3f}, {df_micro}}} = {chi2_R:.4f}')
print(f'    （分隔右侧面积 {alpha_micro/2} 的值）')
print(f'  💡 注意：χ²_L < χ²_R，因为卡方分布是右偏的')

In [ ]:
# ========== 步骤 3: 计算方差的置信区间 ==========
var_lower = (df_micro * s2_micro) / chi2_R
var_upper = (df_micro * s2_micro) / chi2_L

print(f'📊 步骤 3: 计算总体方差 σ² 的 {confidence_level*100:.0f}% 置信区间')
print(f'  公式: (n-1)s²/χ²_R < σ² < (n-1)s²/χ²_L')
print(f'  计算: ({df_micro} × {s2_micro:.4f}) / {chi2_R:.4f} < σ² < ({df_micro} × {s2_micro:.4f}) / {chi2_L:.4f}')
print(f'       {df_micro * s2_micro:.4f} / {chi2_R:.4f} < σ² < {df_micro * s2_micro:.4f} / {chi2_L:.4f}')
print(f'       {var_lower:.4f} < σ² < {var_upper:.4f}')

In [ ]:
# ========== 步骤 4: 计算标准差的置信区间 ==========
std_lower = np.sqrt(var_lower)
std_upper = np.sqrt(var_upper)

print(f'📊 步骤 4: 计算总体标准差 σ 的 {confidence_level*100:.0f}% 置信区间')
print(f'  对方差区间开平方:')
print(f'  √{var_lower:.4f} < σ < √{var_upper:.4f}')
print(f'  {std_lower:.4f} < σ < {std_upper:.4f}')
print(f'  保留1位小数: {std_lower:.1f} bpm < σ < {std_upper:.1f} bpm')

print(f'\n🎯 解读:')
print(f'  我们有 95% 的把握认为，成年女性心率的标准差在 {std_lower:.1f} bpm 和 {std_upper:.1f} bpm 之间')
print(f'  ⚠️ 注意：置信区间不能用 x̄ ± E 的形式表达')

---

## 5. 用 scipy 验证手算结果

In [ ]:
# ========== 用 scipy 验证 ==========
# scipy 没有直接的方差 CI 函数，但我们可以用 chi2 分布计算

# 方法: 使用卡方分布的 PPF 函数
chi2_L_scipy = stats.chi2.ppf(0.025, df_micro)
chi2_R_scipy = stats.chi2.ppf(0.975, df_micro)

var_L_scipy = (df_micro * s2_micro) / chi2_R_scipy
var_U_scipy = (df_micro * s2_micro) / chi2_L_scipy

std_L_scipy = np.sqrt(var_L_scipy)
std_U_scipy = np.sqrt(var_U_scipy)

print('🔬 scipy 验证:')
print(f'  手算 χ²_L = {chi2_L:.6f}')
print(f'  scipy χ²_L = {chi2_L_scipy:.6f}')
print(f'  手算 χ²_R = {chi2_R:.6f}')
print(f'  scipy χ²_R = {chi2_R_scipy:.6f}')
print(f'\n  手算方差 CI = ({var_lower:.6f}, {var_upper:.6f})')
print(f'  scipy 方差 CI = ({var_L_scipy:.6f}, {var_U_scipy:.6f})')
print(f'\n  手算标准差 CI = ({std_lower:.6f}, {std_upper:.6f})')
print(f'  scipy 标准差 CI = ({std_L_scipy:.6f}, {std_U_scipy:.6f})')
print(f'\n  ✅ 验证通过！')

---

## 6. 正态性检验：方差估计的前提条件

In [ ]:
# ========== 正态性检验 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：直方图
ax1 = axes[0]
ax1.hist(heart_rates, bins=8, density=True, alpha=0.6, color='steelblue', edgecolor='white')
x_fit = np.linspace(heart_rates.min() - 5, heart_rates.max() + 5, 100)
y_fit = stats.norm.pdf(x_fit, np.mean(heart_rates), s_micro)
ax1.plot(x_fit, y_fit, 'r-', linewidth=2, label='Fitted Normal')
ax1.set_xlabel('Heart Rate (bpm)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Distribution of Heart Rates', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图：QQ 图
ax2 = axes[1]
stats.probplot(heart_rates, dist='norm', plot=ax2)
ax2.set_title('Normal Q-Q Plot', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Shapiro-Wilk 检验
shapiro_stat, shapiro_p = stats.shapiro(heart_rates)
print(f'\n🔬 Shapiro-Wilk 正态性检验:')
print(f'  检验统计量 W = {shapiro_stat:.4f}')
print(f'  p 值 = {shapiro_p:.4f}')
if shapiro_p > 0.05:
    print(f'  ✅ p > 0.05，不拒绝正态性假设，可以使用卡方方法')
else:
    print(f'  ⚠️ p ≤ 0.05，拒绝正态性假设，卡方方法可能不适用')

---

## 7. 大样本模拟：500 个数据点

In [ ]:
# ========== 大样本模拟 ==========
n_large = 500
mu_true = 70      # 真实均值
sigma_true = 10   # 真实标准差

data_large = np.random.normal(mu_true, sigma_true, n_large)

# 计算统计量
s2_large = np.var(data_large, ddof=1)
s_large = np.std(data_large, ddof=1)
df_large = n_large - 1

# 卡方临界值
chi2_L_large = stats.chi2.ppf(0.025, df_large)
chi2_R_large = stats.chi2.ppf(0.975, df_large)

# 方差置信区间
var_L_large = (df_large * s2_large) / chi2_R_large
var_U_large = (df_large * s2_large) / chi2_L_large

# 标准差置信区间
std_L_large = np.sqrt(var_L_large)
std_U_large = np.sqrt(var_U_large)

print('=' * 60)
print('📋 大样本：心率数据的方差估计')
print('=' * 60)
print(f'\n📊 数据生成过程 (DGP):')
print(f'  分布: Normal(μ={mu_true}, σ={sigma_true})')
print(f'  样本量: {n_large}')

print(f'\n📊 样本统计量:')
print(f'  样本方差 s² = {s2_large:.4f} (真实 σ² = {sigma_true**2})')
print(f'  样本标准差 s = {s_large:.4f} (真实 σ = {sigma_true})')

print(f'\n📊 95% 置信区间:')
print(f'  χ²_L = {chi2_L_large:.4f}, χ²_R = {chi2_R_large:.4f}')
print(f'  方差: {var_L_large:.4f} < σ² < {var_U_large:.4f}')
print(f'  标准差: {std_L_large:.4f} < σ < {std_U_large:.4f}')

print(f'\n🎯 检查:')
if var_L_large <= sigma_true**2 <= var_U_large:
    print(f'  ✅ 真实方差 σ² = {sigma_true**2} 被包含在置信区间内')
else:
    print(f'  ⚠️ 真实方差 σ² = {sigma_true**2} 未被包含在置信区间内')
if std_L_large <= sigma_true <= std_U_large:
    print(f'  ✅ 真实标准差 σ = {sigma_true} 被包含在置信区间内')
else:
    print(f'  ⚠️ 真实标准差 σ = {sigma_true} 未被包含在置信区间内')

---

## 8. 三种估计方法的对比

In [ ]:
# ========== 三种估计方法的对比 ==========
# 使用心率数据
x_bar = np.mean(heart_rates)
se_mean = s_micro / np.sqrt(n_micro)

# 1. 比例的置信区间（如果适用）
# 2. 均值的置信区间（t 分布）
t_crit = stats.t.ppf(0.975, df_micro)
mean_CI_lower = x_bar - t_crit * se_mean
mean_CI_upper = x_bar + t_crit * se_mean

# 3. 标准差的置信区间（卡方分布）

print('=' * 60)
print('📋 三种参数的估计对比（心率数据，95% CI）')
print('=' * 60)

print(f'\n1. 均值 μ 的置信区间（t 分布）:')
print(f'   x̄ = {x_bar:.4f}')
print(f'   t_{{crit}} = {t_crit:.4f}')
print(f'   SE = {se_mean:.4f}')
print(f'   CI: ({mean_CI_lower:.4f}, {mean_CI_upper:.4f})')

print(f'\n2. 方差 σ² 的置信区间（卡方分布）:')
print(f'   s² = {s2_micro:.4f}')
print(f'   CI: ({var_lower:.4f}, {var_upper:.4f})')

print(f'\n3. 标准差 σ 的置信区间（卡方分布）:')
print(f'   s = {s_micro:.4f}')
print(f'   CI: ({std_lower:.4f}, {std_upper:.4f})')

print(f'\n💡 对比要点:')
print(f'   均值的 CI 是对称的（x̄ ± E）')
print(f'   方差/标准差的 CI 是不对称的（因为卡方分布是右偏的）')

---

## 📌 核心概念回顾

### 📌 卡方分布 (Chi-Square Distribution)
- **定义**: $\chi^2 = \frac{(n-1)s^2}{\sigma^2}$ 服从 $df = n-1$ 的卡方分布
- **特征**: 右偏态，非负数
- **含义**: 用于构建方差和标准差的置信区间
- **判断标准**: $df$ 越大，越接近正态分布

### 📌 方差的置信区间
- **公式**: $\frac{(n-1)s^2}{\chi^2_R} < \sigma^2 < \frac{(n-1)s^2}{\chi^2_L}$
- **条件**: 简单随机样本 + **总体服从正态分布**
- **解读**: 「我们有 95% 的把握认为 $\sigma^2$ 在此区间内」
- **特殊性**: 区间不对称，不能用 $\bar{x} \pm E$ 形式

### 📌 标准差的置信区间
- **公式**: 对方差区间开平方
- **含义**: 衡量数据的离散程度

### 📌 正态性要求
- **方差估计**: 对正态性要求**严格**（即使是大样本）
- **均值估计**: 对正态性要求**宽松**（大样本可用 CLT）
- **替代方案**: 如果不满足正态性，使用自助法（7-4 节）

### 🔗 完整流程
```
验证正态性 → 计算s² → 确定df → 查χ²临界值 → 计算方差CI → 开平方得标准差CI
     ↓          ↓        ↓          ↓            ↓              ↓
  QQ图/检验    样本方差   n-1      χ²_L, χ²_R    (n-1)s²/χ²     √(方差CI)
```

### 📝 三种分布的对比

| 分布 | 用途 | 形状 | 自由度 |
|------|------|------|--------|
| 正态 (z) | $\sigma$ 已知时的均值 CI | 对称 | 无 |
| t 分布 | $\sigma$ 未知时的均值 CI | 对称 | $n-1$ |
| 卡方 | 方差/标准差 CI | 右偏 | $n-1$ |

---

## ❌ 常见误区

### ❌ 误区 1: 「方差估计不需要正态性假设」
**✓ 正确理解**: 方差估计对正态性的要求比均值估计**更严格**。即使是大样本，总体也必须服从正态分布。如果数据明显非正态，应使用自助法。

### ❌ 误区 2: 「标准差的 CI 可以用 x̄ ± E 形式」
**✓ 正确理解**: 方差和标准差的置信区间是不对称的（因为卡方分布是右偏的），不能用 $\bar{x} \pm E$ 的形式。书中特别指出：「置信区间可以被表达为 (7.6bpm, 14.2bpm)，但不能使用 s ± E 来表达。」

### ❌ 误区 3: 「样本量越大，方差的 CI 越宽」
**✓ 正确理解**: 样本量越大，方差的 CI 越窄（更精确）。大样本提供了更多信息，减少了不确定性。

### ❌ 误区 4: 「卡方分布和正态分布形状相同」
**✓ 正确理解**: 卡方分布是右偏的（不对称），而正态分布是对称的。只有当自由度很大时，卡方分布才近似正态。

### ❌ 误区 5: 「用样本标准差 s 直接作为 σ 的估计就足够了」
**✓ 正确理解**: 点估计 $s$ 不包含精度信息。置信区间告诉我们 $\sigma$ 可能在什么范围内，以及我们对这个范围有多大的把握。